In [1]:
from dataclasses import dataclass
from typing import Iterator

from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor

from torch import Tensor
import torch


@dataclass(slots=True, frozen=True)
class CifarItem:
    image: Tensor # (3, 32, 32)
    label: Tensor # ()


class CifarInMemoryDataset:
    images: Tensor  # (N, 3, 32, 32)
    labels: Tensor  # (N,)

    def __init__(self, root: str = 'data/'):
        dataset = CIFAR10(root=root, download=True, transform=ToTensor())

        # Preload all data into memory as tensors
        self.images = torch.stack([dataset[i][0] for i in range(len(dataset))])  # (N, 3, 32, 32)
        self.labels = torch.tensor([dataset[i][1] for i in range(len(dataset))])  # (N,)

        # Share memory for multi-process data loading
        self.images.share_memory_()
        self.labels.share_memory_()

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> CifarItem:
        return CifarItem(image=self.images[index], label=self.labels[index])

    def __iter__(self) -> Iterator[CifarItem]:
        for i in range(len(self)):
            yield self[i]

dataset = CifarInMemoryDataset()
print(f'Number of samples in CifarInMemoryDataset: {len(dataset)}')

Number of samples in CifarInMemoryDataset: 50000


In [ ]:
from typing import final
from matplotlib.pylab import f
from pyparsing import C
from apriori.ico.core.operator import IcoOperator
from apriori.ico.core.source import IcoSource
from apriori.ico.utils.data.batcher import IcoBatcher


cifar_items = IcoSource[CifarItem](lambda: iter(dataset), name='CIFAR10 dataset')
batcher = IcoBatcher[CifarItem](batch_size=8)

data_input_flow = cifar_items | batcher
data_input_flow.name = 'CIFAR10 In-Memory Data Input Flow'
data_input_flow.describe()

for item_batch in data_input_flow(None):
    print(f'Number of items in batch: {len(list(item_batch))}')
    break




──────────────────────────── ICO flow description of CIFAR10 In-Memory Data Input Flow ────────────────────────────

CIFAR10 In-Memory Data Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
├── CIFAR10 dataset (source)  () → Iterator[CifarItem]
└── batcher(8) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]

Number of items in batch: 8


In [3]:
from abc import ABC, abstractmethod
from typing import Callable
from torchvision.transforms import functional as F


class CifarTransform(ABC):
    p: float

    def __init__(self, p: float = 1.0):
        self.p = p

    @abstractmethod
    def _image_transform(self, image: Tensor) -> Tensor:
        raise NotImplementedError

    def __call__(self, item: CifarItem) -> CifarItem:
        if torch.rand(1) >= self.p:
            return item

        return CifarItem(
            image=self._image_transform(item.image),
            label=item.label
        )

class HorizontalFlip(CifarTransform):
    def _image_transform(self, image: Tensor) -> Tensor:
        return F.hflip(image)


class VerticalFlip(CifarTransform):
    def _image_transform(self, image: Tensor) -> Tensor:
        return F.vflip(image)


class AdjustBrightness(CifarTransform):

    def __init__(self, p: float = 0.5, factor: float = 0.2):
        super().__init__(p)
        self.factor = factor

    def _image_transform(self, image: Tensor) -> Tensor:
        return F.adjust_brightness(image, self.factor)


class AdjustContrast(CifarTransform):

    def __init__(self, p: float = 0.5, factor: float = 0.2):
        super().__init__(p)
        self.factor = factor

    def _image_transform(self, image: Tensor) -> Tensor:
        return F.adjust_contrast(image, self.factor)


item_aug_flow = (
    IcoOperator(HorizontalFlip())
    | IcoOperator(VerticalFlip())
    | IcoOperator(AdjustBrightness())
    | IcoOperator(AdjustContrast())
)
item_aug_flow.name = "Item augmentation flow"
item_aug_flow.describe()




───────────────────────────────── ICO flow description of Item augmentation flow ──────────────────────────────────

Item augmentation flow (chain)  CifarItem → CifarItem
├── IcoOperator | IcoOperator | IcoOperator (chain)  CifarItem → CifarItem
│   ├── IcoOperator | IcoOperator (chain)  CifarItem → CifarItem
│   │   ├── IcoOperator (operator)  CifarItem → CifarItem
│   │   └── IcoOperator (operator)  CifarItem → CifarItem
│   └── IcoOperator (operator)  CifarItem → CifarItem
└── IcoOperator (operator)  CifarItem → CifarItem

In [4]:
from apriori.ico.core.streamline import IcoStreamline


item_aug_flow = IcoStreamline(
    [
        HorizontalFlip(),
        VerticalFlip(),
        AdjustBrightness(),
        AdjustContrast()
    ]
)
item_aug_flow.name = "Item augmentation flow"
item_aug_flow.describe()

───────────────────────────────── ICO flow description of Item augmentation flow ──────────────────────────────────

Item augmentation flow (streamline)  CifarItem → CifarItem
├── IcoOperator (operator)  CifarItem → CifarItem
├── IcoOperator (operator)  CifarItem → CifarItem
├── IcoOperator (operator)  CifarItem → CifarItem
└── IcoOperator (operator)  CifarItem → CifarItem

In [ ]:
from typing import TypeAlias

CifarItemBatch: TypeAlias = Iterator[CifarItem]

@final
class CifarBatch:
    __slots__ = ('images', 'labels')

    images: Tensor  # (B, 3, 32, 32)
    labels: Tensor  # (B,)

    def __init__(self, items: CifarItemBatch):
        items_list = list(items)
        self.images = torch.stack([item.image for item in items_list])  # (B, 3, 32, 32)
        self.labels = torch.tensor([item.label for item in items_list])  # (B,)


def collate(items: CifarItemBatch) -> CifarBatch:
    return CifarBatch(items)

def to_shared_memory(batch: CifarBatch) -> CifarBatch:
    batch.images.share_memory_()
    batch.labels.share_memory_()
    return batch

aug_flow = item_aug_flow.iterate() | IcoOperator(collate) | IcoOperator(to_shared_memory)
aug_flow.name = "Full Augmentation flow"
aug_flow.describe()

───────────────────────────────── ICO flow description of Full Augmentation flow ──────────────────────────────────

Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
├── iterate(Item augmentation flow)] | collate (chain)  Iterator[CifarItem] → CifarBatch
│   ├── iterate(Item augmentation flow)] (iterate)  Iterator[CifarItem] → Iterator[CifarItem]
│   │   └── Item augmentation flow (streamline)  CifarItem → CifarItem
│   │       ├── IcoOperator (operator)  CifarItem → CifarItem
│   │       ├── IcoOperator (operator)  CifarItem → CifarItem
│   │       ├── IcoOperator (operator)  CifarItem → CifarItem
│   │       └── IcoOperator (operator)  CifarItem → CifarItem
│   └── collate (operator)  Iterator[CifarItem] → CifarBatch
└── to_shared_memory (operator)  CifarBatch → CifarBatch

In [6]:
full_data_flow = data_input_flow | aug_flow.iterate()
full_data_flow.name = "Full Data Flow with Augmentation"
full_data_flow.describe()

──────────────────────────── ICO flow description of Full Data Flow with Augmentation ─────────────────────────────

Full Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
├── CIFAR10 In-Memory Data Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│   ├── CIFAR10 dataset (source)  () → Iterator[CifarItem]
│   └── batcher(8) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
└── iterate(Full Augmentation flow)] (iterate)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
    └── Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
        ├── iterate(Item augmentation flow)] | collate (chain)  Iterator[CifarItem] → CifarBatch
        │   ├── iterate(Item augmentation flow)] (iterate)  Iterator[CifarItem] → Iterator[CifarItem]
        │   │   └── Item augmentation flow (streamline)  CifarItem → CifarItem
        │   │       ├── IcoOperator (operator)  CifarItem → CifarItem
        │   │       ├── IcoOperator (operator)  CifarItem → CifarItem
        │   │       ├── IcoOperator (operator)  CifarItem → CifarItem
        │   │       └── IcoOperator (operator)  CifarItem → CifarItem
        │   └── collate (operator)  Iterator[CifarItem] → CifarBatch
        └── to_shared_memory (operator)  CifarBatch → CifarBatch

In [7]:
for batch in full_data_flow(None):
    print(f'Batch images shape: {batch.images.shape}, Batch labels shape: {batch.labels.shape}')
    break

Batch images shape: torch.Size([8, 3, 32, 32]), Batch labels shape: torch.Size([8])


In [ ]:
from apriori.ico.core.async_stream import IcoAsyncStream
from apriori.ico.runtime.agent.mp_process.mp_process import MPProcess
from examples.visual.cifar.data_flow import create_augmentation_flow, CifarBatch, create_data_input_flow

data_input_flow = create_data_input_flow(batch_size=8)
num_workers = 4
workers_pool = [MPProcess(create_augmentation_flow) for _ in range(num_workers)]

async_stream = IcoAsyncStream(workers_pool, ordered=False, name="Async Augmentation Stream")

parallel_data_flow = data_input_flow | async_stream
parallel_data_flow.name = "Parallel Data Flow with Augmentation"
parallel_data_flow.describe()

────────────────────────── ICO flow description of Parallel Data Flow with Augmentation ───────────────────────────

Parallel Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
├── CIFAR10 In-Memory Data Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│   ├── CIFAR10 dataset (source)  () → Iterator[CifarItem]
│   └── batcher(8) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
└── Async Augmentation Stream (async_stream)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
    ├── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch
    ├── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch
    ├── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch
    └── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch

In [ ]:
runtime = parallel_data_flow.runtime()
runtime.activate()
parallel_data_flow.describe()


────────────────────────── ICO flow description of Parallel Data Flow with Augmentation ───────────────────────────

Parallel Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
├── CIFAR10 In-Memory Data Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│   ├── CIFAR10 dataset (source)  () → Iterator[CifarItem]
│   └── batcher(8) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
└── Async Augmentation Stream (async_stream)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
    ├── mp_process (operator) [ready]  Iterator[CifarItem] → CifarBatch
    ├── mp_process (operator) [ready]  Iterator[CifarItem] → CifarBatch
    ├── mp_process (operator) [ready]  Iterator[CifarItem] → CifarBatch
    └── mp_process (operator) [ready]  Iterator[CifarItem] → CifarBatch

Process SpawnProcess-1:
Traceback (most recent call last):
  File "/Users/shum/dev/apriori/ico/src/apriori/ico/runtime/agent/mp_process/mp_process_agent.py", line 61, in _run_loop
    result = wait_for_item(
  File "/Users/shum/dev/apriori/ico/src/apriori/ico/core/runtime/channel/utils.py", line 22, in wait_for_item
    input = endpoint.receive()
  File "/Users/shum/dev/apriori/ico/src/apriori/ico/runtime/channel/mp_queue/receive_endpoint.py", line 53, in receive
    message = self._main_queue.get(timeout=self._timeout)
  File "/Users/shum/.pyenv/versions/3.10.14/lib/python3.10/multiprocessing/queues.py", line 114, in get
    raise Empty
_queue.Empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/shum/dev/apriori/ico/src/apriori/ico/runtime/channel/mp_queue/send_endpoint.py", line 74, in _wait_for_ack
    ack_message = self._ack_queue.get(timeout=timeout)
  File "/Users/shum/.pyenv/versions/3.10.14/lib/python3.10/